In [ ]:
!pip install PyMuPDF bitsandbytes accelerate transformers datasets -q

In [ ]:
import json
import re
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq

## Define Constants



In [ ]:
BASE_MODEL = 't5-small'
JSON_DATA_PATH = '/content/marketing_summarization_dataset.jsonl'

##
Load the Dataset



In [ ]:
from datasets import load_dataset

data = load_dataset('json', data_files=JSON_DATA_PATH, split='train')
print(f"Dataset loaded with {len(data)} examples.")
print("First example:\n", data[0])

Dataset loaded with 12 examples.
First example:
 {'messages': [{'role': 'system', 'content': 'You are a marketing expert assistant. Summarize marketing content clearly, concisely, and accurately, highlighting key insights, strategies, and actionable takeaways.'}, {'role': 'user', 'content': "Summarize the following email campaign report:\n\nCampaign Name: Spring Sale 2024\nSent: 85,000 emails\nOpen Rate: 24.3%\nClick-Through Rate: 6.8%\nConversion Rate: 2.1%\nRevenue Generated: $142,500\nTop Performing Subject Line: 'Last Chance: 40% Off Ends Tonight'\nBest Performing Segment: Loyal customers (3+ purchases)\nUnsubscribes: 320\nBounces: 1,200\nTop Clicked Link: Spring Collection CTA button\nDevice Split: 68% mobile, 32% desktop\nBest Send Time: Tuesday 10am"}, {'role': 'assistant', 'content': "**Spring Sale 2024 Email Campaign Summary**\n\nThe campaign reached 85,000 subscribers and delivered strong results across key metrics. With a 24.3% open rate and 6.8% CTR, both figures exceed typ

In [ ]:
# Function to extract user prompt and assistant response from a conversation
def extract_summarization_pair(conversation):
    user_prompt = None
    assistant_response = None
    for message in conversation:
        if message['role'] == 'user':
            user_prompt = message['content']
        elif message['role'] == 'assistant':
            assistant_response = message['content']
    return {"user_prompt": user_prompt, "assistant_response": assistant_response}

# Apply the function to your dataset
summarization_data = [extract_summarization_pair(conv) for conv in data['messages']]

# Display the first few extracted pairs
print("Extracted summarization pairs. First 3 pairs:")
for i, item in enumerate(summarization_data[:3]):
    print(f"--- Pair {i+1} ---")
    print(f"User Prompt: {item['user_prompt']}\n")
    print(f"Assistant Response (Summary): {item['assistant_response']}\n")

Extracted summarization pairs. First 3 pairs:
--- Pair 1 ---
User Prompt: Summarize the following email campaign report:

Campaign Name: Spring Sale 2024
Sent: 85,000 emails
Open Rate: 24.3%
Click-Through Rate: 6.8%
Conversion Rate: 2.1%
Revenue Generated: $142,500
Top Performing Subject Line: 'Last Chance: 40% Off Ends Tonight'
Best Performing Segment: Loyal customers (3+ purchases)
Unsubscribes: 320
Bounces: 1,200
Top Clicked Link: Spring Collection CTA button
Device Split: 68% mobile, 32% desktop
Best Send Time: Tuesday 10am

Assistant Response (Summary): **Spring Sale 2024 Email Campaign Summary**

The campaign reached 85,000 subscribers and delivered strong results across key metrics. With a 24.3% open rate and 6.8% CTR, both figures exceed typical industry benchmarks, generating $142,500 in revenue at a 2.1% conversion rate.

**Top Insights:**
- Urgency-driven subject lines drove opens — 'Last Chance: 40% Off Ends Tonight' was the standout performer
- Loyal customers (3+ purchase

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# T5 tokenizers usually have a pad_token defined by default.
# Ensure it's explicitly set if None, though usually not an issue for T5.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set tokenizer.pad_token to: {tokenizer.pad_token}")

def tokenize_function(examples):
    # For Seq2Seq models, we tokenize input and target separately.
    # The 'text_target' argument automatically handles the labels for the decoder.
    # We also add the 'summarize:' prefix often used with T5 models for summarization tasks.
    inputs = [f"summarize: {prompt}" for prompt in examples['user_prompt']]
    targets = examples['assistant_response']

    model_inputs = tokenizer(
        inputs,
        max_length=512, # Max length for input sequence
        truncation=True,
    )

    # Setup the tokenizer for targets
    labels = tokenizer(
        targets,
        max_length=128, # Max length for target sequence (summary)
        truncation=True,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Convert the list of dictionaries to a Dataset object for easier processing
summarization_dataset = Dataset.from_list(summarization_data)

tokenized_datasets = summarization_dataset.map(tokenize_function, batched=True, remove_columns=summarization_dataset.column_names)

print("First tokenized example:")
print(tokenized_datasets[0])
print("\nDecoded input_ids (user_prompt):")
print(tokenizer.decode(tokenized_datasets[0]['input_ids']))
print("\nDecoded labels (assistant_response):")
print(tokenizer.decode(tokenized_datasets[0]['labels']))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

First tokenized example:
{'input_ids': [21603, 10, 12198, 1635, 1737, 8, 826, 791, 2066, 934, 10, 18409, 5570, 10, 4328, 5774, 460, 2266, 4892, 17, 10, 505, 5898, 7594, 2384, 13002, 10, 997, 5, 5170, 2001, 18, 11889, 4607, 13002, 10, 4357, 5953, 1193, 8674, 13002, 10, 1682, 4704, 19764, 6939, 4094, 10, 26845, 22092, 2224, 3, 26656, 19237, 4919, 10, 3, 31, 3612, 7, 17, 8622, 10, 13152, 4395, 3720, 7, 24951, 31, 1648, 3, 26656, 15696, 297, 10, 1815, 63, 138, 722, 6918, 1220, 9701, 61, 597, 7304, 19308, 7, 10, 3, 15003, 272, 7906, 7, 10, 1914, 3632, 2224, 2001, 15, 26, 7505, 10, 4328, 6767, 205, 3221, 2218, 15511, 23575, 10, 431, 5953, 1156, 6, 220, 5406, 6555, 1648, 9384, 2900, 10, 2818, 335, 265, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

print(f"Successfully loaded {BASE_MODEL} model for Sequence-to-Sequence LM.")

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Successfully loaded t5-small model for Sequence-to-Sequence LM.


In [ ]:
from transformers import DataCollatorForSeq2Seq

# The DataCollatorForSeq2Seq automatically handles padding and shifts labels for T5.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=tokenizer.pad_token_id)

print("Data collator initialized.")

Data collator initialized.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",             # output directory for model checkpoints and logs
    num_train_epochs=3,                 # total number of training epochs
    per_device_train_batch_size=4,      # batch size per device during training
    per_device_eval_batch_size=4,       # batch size per device during evaluation
    warmup_steps=500,                   # number of warmup steps for learning rate scheduler
    weight_decay=0.01,                  # strength of weight decay
    logging_dir="./logs",               # directory for storing logs
    logging_steps=10,                   # Log every N updates steps
    # evaluation_strategy="epoch",        # Removed to fix error
    # save_strategy="epoch",              # Removed to fix error
    # load_best_model_at_end=True,        # Removed if save_strategy is removed
    # metric_for_best_model="eval_loss",  # Removed if evaluation is not integrated
    # greater_is_better=False,            # Removed if evaluation is not integrated
    report_to="none"                    # Disable integrations like W&B for simplicity
)

print("Training arguments defined.")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments defined.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,   # Using the full dataset as training set for simplicity here
    eval_dataset=tokenized_datasets,    # Using the full dataset as eval set for simplicity (for real training, use a separate validation set)
    # tokenizer=tokenizer,              # Removed this line to fix the TypeError
    data_collator=data_collator,
)

print("Trainer initialized. Ready to fine-tune.")

Trainer initialized. Ready to fine-tune.


In [ ]:
# To start the training process:
trainer.train()

# After training, you can save your fine-tuned model
# trainer.save_model("./fine_tuned_t5_summarizer")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9, training_loss=5.339006635877821, metrics={'train_runtime': 105.5076, 'train_samples_per_second': 0.341, 'train_steps_per_second': 0.085, 'total_flos': 2778736361472.0, 'train_loss': 5.339006635877821, 'epoch': 3.0})

In [ ]:
trainer.save_model("./fine_tuned_t5_summarizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import shutil

# Path to the directory where the model is saved
model_output_dir = './fine_tuned_t5_summarizer'

# Name for the zip file (without .zip extension)
zip_file_name = 'fine_tuned_t5_summarizer'

# Create the zip archive
shutil.make_archive(zip_file_name, 'zip', model_output_dir)

print(f"Model saved as {zip_file_name}.zip")

Model saved as fine_tuned_t5_summarizer.zip


In [ ]:
!pip install PyMuPDF bitsandbytes accelerate transformers datasets peft -q

In [ ]:
import json
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# Configure LoRA
lora_config = LoraConfig(
    r=8,  # LoRA attention dimension
    lora_alpha=32,  # Alpha parameter for LoRA scaling
    target_modules=["q", "v"], # Modules to apply LoRA to, typically attention query and value layers
    bias="none", # Bias type for LoRA layers
    task_type=TaskType.SEQ_2_SEQ_LM # Task type for the model
)

# Wrap the base model with LoRA
model = get_peft_model(model, lora_config)

print(f"Successfully loaded {BASE_MODEL} model for Sequence-to-Sequence LM and applied LoRA.")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Successfully loaded t5-small model for Sequence-to-Sequence LM and applied LoRA.
trainable params: 294,912 || all params: 60,801,536 || trainable%: 0.4850
